In [1]:
# CELL 1 — imports and processing configuration
from pathlib import Path
from collections import defaultdict
import json
import re
import time
import hashlib
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 80)

# Select one export version per assay family for the primary integration table.
# Options: "prefer_transformed" or "prefer_submitted".
# All versions are still retained in osd575_measurements_all_sources.csv.
PRIMARY_SOURCE_POLICY = "prefer_transformed"

# Avoid accidentally loading very large unrelated files.
MAX_FILE_SIZE_MB = 100

print("Imports ready")
print("pandas:", pd.__version__, "| numpy:", np.__version__)
print("Primary source policy:", PRIMARY_SOURCE_POLICY)

Imports ready
pandas: 3.1.1 | numpy: 2.4.6
Primary source policy: prefer_transformed


d:\Miniconda\envs\astronaut-ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# CELL 2 — locate workstream and define project-level paths
cwd = Path.cwd().resolve()
search_paths = [cwd, *cwd.parents]

# Optional override is useful for testing in a temporary folder.
import os
root_override = os.environ.get("OSD575_WORKSTREAM_ROOT")

if root_override:
    WORKSTREAM_ROOT = Path(root_override).expanduser().resolve()
else:
    WORKSTREAM_ROOT = next(
        (p for p in search_paths if p.name.lower() == "ml-workstream-nasa"),
        None
    )
    if WORKSTREAM_ROOT is None:
        parent = next((p for p in search_paths if (p / "ml-workstream-nasa").is_dir()), None)
        if parent is not None:
            WORKSTREAM_ROOT = parent / "ml-workstream-nasa"

if WORKSTREAM_ROOT is None or not WORKSTREAM_ROOT.is_dir():
    raise FileNotFoundError(
        "Could not locate ml-workstream-nasa. Open Jupyter from the workstream "
        "or a child folder, or set OSD575_WORKSTREAM_ROOT."
    )

RAW_DIR = WORKSTREAM_ROOT / "data" / "raw" / "OSD-575"
INTERIM_DIR = WORKSTREAM_ROOT / "data" / "interim"
PROCESSED_DIR = WORKSTREAM_ROOT / "data" / "processed"
REPORTS_DIR = WORKSTREAM_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
TABLES_DIR = REPORTS_DIR / "tables"

for folder in [INTERIM_DIR, PROCESSED_DIR, REPORTS_DIR, FIGURES_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

if not RAW_DIR.is_dir():
    raise FileNotFoundError(f"Raw OSD-575 folder not found: {RAW_DIR}")

print("Workstream:", WORKSTREAM_ROOT)
print("Raw files:", RAW_DIR)
print("Processed outputs:", PROCESSED_DIR)

Workstream: D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-nasa
Raw files: D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-nasa\data\raw\OSD-575
Processed outputs: D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-nasa\data\processed


In [3]:
# CELL 3 — discover assay result files (skip ISA metadata text documents)
SUPPORTED_SUFFIXES = {".csv", ".tsv", ".txt", ".xlsx", ".xlsm", ".xls"}
ISA_PREFIXES = ("a_", "i_", "s_")
ASSAY_NAME_TOKENS = ["lsd", "panel", "metabolic", "immuno", "cardio", "eve", "alamar", "hd71", "hdcvd9"]

all_files = sorted(p for p in RAW_DIR.rglob("*") if p.is_file())
file_rows = []
assay_paths = []

for path in all_files:
    name_lower = path.name.lower()
    is_isa = name_lower.startswith(ISA_PREFIXES)
    is_table = path.suffix.lower() in SUPPORTED_SUFFIXES
    is_candidate = is_table and not is_isa and any(token in name_lower for token in ASSAY_NAME_TOKENS)
    if is_candidate:
        assay_paths.append(path)
    file_rows.append({
        "file_name": path.name,
        "relative_path": str(path.relative_to(RAW_DIR)),
        "extension": path.suffix.lower() or "[none]",
        "size_bytes": path.stat().st_size,
        "is_isa_metadata": is_isa,
        "is_assay_candidate": is_candidate,
    })

file_inventory = pd.DataFrame(file_rows)
file_inventory.to_csv(TABLES_DIR / "osd575_processing_file_inventory.csv", index=False)
display(file_inventory)

if not assay_paths:
    raise RuntimeError(
        "No assay-result files found. Check filenames/extensions in "
        "reports/tables/osd575_processing_file_inventory.csv."
    )

print(f"\nDiscovered {len(assay_paths)} assay result files from {len(all_files)} total files.")

,file_name,relative_path,extension,size_bytes,is_isa_metadata,is_assay_candidate
0,a_OSD-575_analytical-chemistry_comprehensive-m...,a_OSD-575_analytical-chemistry_comprehensive-m...,.txt,7973,True,False
1,a_OSD-575_immunoassay_multiplex_Alamar Bioscie...,a_OSD-575_immunoassay_multiplex_Alamar Bioscie...,.txt,10917,True,False
2,a_OSD-575_immunoassay_multiplex_Eve Technologi...,a_OSD-575_immunoassay_multiplex_Eve Technologi...,.txt,11665,True,False
3,a_OSD-575_immunoassay_multiplex_Eve Technologi...,a_OSD-575_immunoassay_multiplex_Eve Technologi...,.txt,13037,True,False
4,i_Investigation.txt,i_Investigation.txt,.txt,13753,True,False
5,LSDS-8_Comprehensive_Metabolic_Panel_CMP.uploa...,LSDS-8_Comprehensive_Metabolic_Panel_CMP.uploa...,.csv,25275,False,True
6,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,.csv,8506,False,True
7,LSDS-8_Multiplex_serum.cardiovascular.EvePanel...,LSDS-8_Multiplex_serum.cardiovascular.EvePanel...,.csv,15241,False,True
8,LSDS-8_Multiplex_serum.immune.AlamarPanel_SUBM...,LSDS-8_Multiplex_serum.immune.AlamarPanel_SUBM...,.xlsx,322799,False,True
9,LSDS-8_Multiplex_serum.immune.AlamarPanel_TRAN...,LSDS-8_Multiplex_serum.immune.AlamarPanel_TRAN...,.csv,141783,False,True



Discovered 8 assay result files from 14 total files.


In [4]:
# CELL 4 — reliable table readers for CSV/TSV/TXT and Excel workbooks

def read_delimited(path):
    errors = []
    for encoding in ["utf-8-sig", "utf-8", "cp1252", "latin-1"]:
        try:
            frame = pd.read_csv(path, encoding=encoding, sep=None, engine="python")
            if frame.shape[1] == 1:
                for sep in [",", "\t", ";"]:
                    retry = pd.read_csv(path, encoding=encoding, sep=sep)
                    if retry.shape[1] > 1:
                        frame = retry
                        break
            return frame
        except Exception as exc:
            errors.append(f"{encoding}: {type(exc).__name__}: {exc}")
    raise RuntimeError(" | ".join(errors))


def read_assay_file(path):
    suffix = path.suffix.lower()
    if suffix in {".csv", ".tsv", ".txt"}:
        return {path.stem: read_delimited(path)}
    if suffix in {".xlsx", ".xlsm", ".xls"}:
        sheets = pd.read_excel(path, sheet_name=None)
        return {f"{path.stem}__{sheet}": frame for sheet, frame in sheets.items()}
    return {}

assay_tables = {}
table_meta = {}
load_rows = []
start = time.perf_counter()

for idx, path in enumerate(tqdm(assay_paths, desc="Reading assay files"), start=1):
    size_mb = path.stat().st_size / (1024 ** 2)
    if size_mb > MAX_FILE_SIZE_MB:
        load_rows.append({"file": path.name, "status": "skipped_size_limit", "detail": f"{size_mb:.2f} MB"})
        continue
    try:
        loaded = read_assay_file(path)
        for table_key, frame in loaded.items():
            frame = frame.loc[:, ~frame.columns.astype(str).str.match(r"^Unnamed", case=False)].copy()
            frame.columns = [str(c).strip() for c in frame.columns]
            assay_tables[table_key] = frame
            table_meta[table_key] = {"source_file": path.name, "source_path": str(path), "sheet": table_key.split("__", 1)[1] if "__" in table_key else ""}
            load_rows.append({"file": path.name, "table": table_key, "status": "loaded", "rows": len(frame), "columns": len(frame.columns), "detail": ""})
            print(f"[{idx}/{len(assay_paths)}] {table_key}: {frame.shape}")
    except Exception as exc:
        load_rows.append({"file": path.name, "status": "error", "detail": f"{type(exc).__name__}: {exc}"})
        print(f"ERROR loading {path.name}: {exc}")

load_log = pd.DataFrame(load_rows)
load_log.to_csv(TABLES_DIR / "osd575_processing_load_log.csv", index=False)
print(f"\nLoaded {len(assay_tables)} table(s) in {time.perf_counter() - start:.2f}s.")
if not assay_tables:
    raise RuntimeError("No assay tables loaded successfully. Review the processing load log.")

Reading assay files: 100%|██████████| 8/8 [00:00<00:00, 196.51it/s]

[1/8] LSDS-8_Comprehensive_Metabolic_Panel_CMP.upload_SUBMITTED: (531, 9)
[2/8] LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANSFORMED: (28, 58)
[3/8] LSDS-8_Multiplex_serum.cardiovascular.EvePanel_SUBMITTED: (252, 8)
ERROR loading LSDS-8_Multiplex_serum.immune.AlamarPanel_SUBMITTED.xlsx: `Import openpyxl` failed.  Use pip, conda, or your preferred package management tool to install the openpyxl package.
[5/8] LSDS-8_Multiplex_serum.immune.AlamarPanel_TRANSFORMED: (27, 407)
[6/8] LSDS-8_Multiplex_serum.immune.EvePanel_SUBMITTED: (1988, 8)
[7/8] LSDS-8_Multiplex_serum_cardiovascular_EvePanel_TRANSFORMED: (28, 19)
[8/8] LSDS-8_Multiplex_serum_immune_EvePanel_TRANSFORMED: (28, 143)

Loaded 7 table(s) in 0.06s.


In [5]:
# CELL 5 — classify assay family and source version from file provenance

def classify_source(source_name):
    text = str(source_name).lower()
    if "cmp" in text or "comprehensive_metabolic" in text:
        family = "comprehensive_metabolic_panel"
        platform = "Quest Diagnostics CMP"
    elif "hdcvd9" in text or "cardiovascular" in text or "cvd" in text:
        family = "cardiovascular_multiplex"
        platform = "Eve Technologies HDCVD9"
    elif "alamar" in text:
        family = "immune_alamar_multiplex"
        platform = "Alamar Biosciences"
    elif "hd71" in text or "evepanel" in text or "eve_panel" in text or "immune" in text:
        family = "immune_eve_multiplex"
        platform = "Eve Technologies HD71"
    else:
        family = "unknown_assay_family"
        platform = "unknown"

    if "transformed" in text:
        version = "TRANSFORMED"
    elif "submitted" in text or "upload" in text:
        version = "SUBMITTED"
    else:
        version = "UNSPECIFIED"
    return family, platform, version

provenance_rows = []
for table_name, frame in assay_tables.items():
    meta = table_meta[table_name]
    family, platform, version = classify_source(meta["source_file"])
    provenance_rows.append({
        "table": table_name,
        "source_file": meta["source_file"],
        "sheet": meta["sheet"],
        "assay_family": family,
        "assay_platform": platform,
        "file_version": version,
        "rows": len(frame),
        "columns": len(frame.columns),
    })

provenance_df = pd.DataFrame(provenance_rows)
provenance_df.to_csv(TABLES_DIR / "osd575_processing_provenance.csv", index=False)
display(provenance_df.sort_values(["assay_family", "file_version", "source_file"]))

,table,source_file,sheet,assay_family,assay_platform,file_version,rows,columns
2,LSDS-8_Multiplex_serum.cardiovascular.EvePanel...,LSDS-8_Multiplex_serum.cardiovascular.EvePanel...,,cardiovascular_multiplex,Eve Technologies HDCVD9,SUBMITTED,252,8
5,LSDS-8_Multiplex_serum_cardiovascular_EvePanel...,LSDS-8_Multiplex_serum_cardiovascular_EvePanel...,,cardiovascular_multiplex,Eve Technologies HDCVD9,TRANSFORMED,28,19
0,LSDS-8_Comprehensive_Metabolic_Panel_CMP.uploa...,LSDS-8_Comprehensive_Metabolic_Panel_CMP.uploa...,,comprehensive_metabolic_panel,Quest Diagnostics CMP,SUBMITTED,531,9
1,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,,comprehensive_metabolic_panel,Quest Diagnostics CMP,TRANSFORMED,28,58
3,LSDS-8_Multiplex_serum.immune.AlamarPanel_TRAN...,LSDS-8_Multiplex_serum.immune.AlamarPanel_TRAN...,,immune_alamar_multiplex,Alamar Biosciences,TRANSFORMED,27,407
4,LSDS-8_Multiplex_serum.immune.EvePanel_SUBMITTED,LSDS-8_Multiplex_serum.immune.EvePanel_SUBMITT...,,immune_eve_multiplex,Eve Technologies HD71,SUBMITTED,1988,8
6,LSDS-8_Multiplex_serum_immune_EvePanel_TRANSFO...,LSDS-8_Multiplex_serum_immune_EvePanel_TRANSFO...,,immune_eve_multiplex,Eve Technologies HD71,TRANSFORMED,28,143


In [6]:
# CELL 6 — identify source columns and normalize one assay table to long format
# This is deliberately schema-aware and conservative. It does not guess clinical units.

SUBJECT_RE = re.compile(r"(subject|participant|crew|astronaut).*(id|identifier)?|factor.?value.*subject", re.I)
SAMPLE_RE = re.compile(r"(sample|specimen|biospecimen).*(id|name|identifier)?|aliquot", re.I)
TIME_RE = re.compile(r"(time.?point|time.?collected|collection.?time|visit|flight.?day|factor.?value.*time|timepoint)", re.I)
ANALYTE_RE = re.compile(r"^(analyte(?:\s*name)?|biomarker(?:\s*name)?|protein(?:\s*name)?|cytokine(?:\s*name)?|marker(?:\s*name)?|test\s*name|component|parameter|target(?:\s*name)?|assay\s*name)$", re.I)
UNIT_RE = re.compile(r"(^|[^a-z])(unit|units|uom)([^a-z]|$)", re.I)
VALUE_RE = re.compile(r"(^|[^a-z])(result|value|concentration|measurement|intensity|signal|abundance|level)([^a-z]|$)", re.I)
NON_MEASUREMENT_RE = re.compile(
    r"(subject|participant|crew|astronaut|sample|specimen|biospecimen|aliquot|time|visit|date|day|factor|"
    r"unit|uom|reference|normal.?range|range|lower|upper|low|high|flag|comment|interpret|"
    r"batch|plate|well|replicate|run.?id|record.?id|row.?id|age|sex|gender|barcode|accession)", re.I
)
SUBJECT_TOKEN_RE = re.compile(r"\bC00[1-4]\b", re.I)
TIMEPOINT_TOKEN_RE = re.compile(r"\b(?:L\s*[-+]\s*\d+|R\s*\+\s*\d+|pre.?flight|post.?flight|in.?flight|baseline)\b", re.I)


def first_matching_column(frame, pattern, preferred=()):
    cols = list(frame.columns)
    lower = {str(c).strip().lower(): c for c in cols}
    for exact in preferred:
        if exact.lower() in lower:
            return lower[exact.lower()]
    matches = [c for c in cols if pattern.search(str(c))]
    return matches[0] if len(matches) == 1 else (matches[0] if matches else None)


def numeric_candidate(series):
    converted = pd.to_numeric(series, errors="coerce")
    nonempty = series.notna().sum()
    if nonempty == 0:
        return 0.0, converted
    return float(converted.notna().sum() / nonempty), converted


def extract_subject_from_sample(value):
    if pd.isna(value):
        return None
    match = SUBJECT_TOKEN_RE.search(str(value))
    return match.group(0).upper() if match else None


def normalize_one_table(table_name, frame):
    meta = table_meta[table_name]
    family, platform, version = classify_source(meta["source_file"])

    subject_col = first_matching_column(frame, SUBJECT_RE, ["subject_id", "participant_id", "crew_id"])
    sample_col = first_matching_column(frame, SAMPLE_RE, ["sample_id", "sample_name", "specimen_id"])
    time_col = first_matching_column(frame, TIME_RE, ["timepoint", "time_point", "collection_timepoint", "visit"])
    analyte_col = first_matching_column(frame, ANALYTE_RE, ["analyte", "biomarker", "protein", "test_name", "parameter"])
    unit_col = first_matching_column(frame, UNIT_RE, ["unit", "units", "uom"])
    value_col = first_matching_column(frame, VALUE_RE, ["result", "value", "concentration", "measurement"])

    # If a named result/value column was not recognized, select a highly numeric
    # column only when an analyte field is present (long-format table).
    if analyte_col is not None and value_col is None:
        excluded = {subject_col, sample_col, time_col, analyte_col, unit_col}
        candidates = []
        for col in frame.columns:
            if col in excluded or NON_MEASUREMENT_RE.search(str(col)):
                continue
            ratio, _ = numeric_candidate(frame[col])
            if ratio >= 0.5:
                candidates.append((ratio, col))
        if candidates:
            value_col = sorted(candidates, reverse=True, key=lambda x: x[0])[0][1]

    records = []
    layout = "long" if analyte_col is not None and value_col is not None else "wide"

    for pos, (source_index, row) in enumerate(frame.iterrows(), start=2):
        subject_raw = row[subject_col] if subject_col is not None else None
        sample_raw = row[sample_col] if sample_col is not None else None
        subject_id = str(subject_raw).strip() if pd.notna(subject_raw) else None
        if not subject_id:
            subject_id = extract_subject_from_sample(sample_raw)
        timepoint_raw = row[time_col] if time_col is not None else None
        timepoint = str(timepoint_raw).strip() if pd.notna(timepoint_raw) else None
        sample_id = str(sample_raw).strip() if pd.notna(sample_raw) else None

        if layout == "long":
            raw_value = row[value_col]
            if pd.isna(raw_value):
                continue
            analyte = str(row[analyte_col]).strip() if pd.notna(row[analyte_col]) else None
            unit = str(row[unit_col]).strip() if unit_col is not None and pd.notna(row[unit_col]) else None
            parsed_value = pd.to_numeric(pd.Series([raw_value]), errors="coerce").iloc[0]
            records.append({
                "study_id": "OSD-575", "assay_family": family, "assay_platform": platform,
                "analyte": analyte, "value": parsed_value,
                "raw_result_text": str(raw_value), "unit": unit,
                "subject_id": subject_id, "sample_id": sample_id,
                "timepoint": timepoint, "source_version": version,
                "source_file": meta["source_file"], "source_table": table_name,
                "source_sheet": meta["sheet"], "source_row_number": pos,
                "source_measurement_column": str(value_col),
                "normalization_layout": layout,
                "numeric_parse_status": "numeric" if pd.notna(parsed_value) else "non_numeric_or_censored",
            })
        else:
            excluded_cols = {subject_col, sample_col, time_col}
            excluded_cols |= {c for c in frame.columns if UNIT_RE.search(str(c)) or NON_MEASUREMENT_RE.search(str(c))}
            for col in frame.columns:
                if col in excluded_cols:
                    continue
                raw_value = row[col]
                if pd.isna(raw_value):
                    continue
                ratio, _ = numeric_candidate(frame[col])
                # Only export columns that look like numeric measurement fields.
                if ratio < 0.5:
                    continue
                parsed_value = pd.to_numeric(pd.Series([raw_value]), errors="coerce").iloc[0]
                col_text = str(col).strip()
                unit_match = re.search(r"(?:\(|\[)([^()\[\]]+)(?:\)|\])\s*$", col_text)
                unit = unit_match.group(1).strip() if unit_match else None
                analyte = re.sub(r"\s*(?:\([^()]*\)|\[[^\[\]]*\])\s*$", "", col_text).strip()
                records.append({
                    "study_id": "OSD-575", "assay_family": family, "assay_platform": platform,
                    "analyte": analyte, "value": parsed_value,
                    "raw_result_text": str(raw_value), "unit": unit,
                    "subject_id": subject_id, "sample_id": sample_id,
                    "timepoint": timepoint, "source_version": version,
                    "source_file": meta["source_file"], "source_table": table_name,
                    "source_sheet": meta["sheet"], "source_row_number": pos,
                    "source_measurement_column": col_text,
                    "normalization_layout": layout,
                    "numeric_parse_status": "numeric" if pd.notna(parsed_value) else "non_numeric_or_censored",
                })

    result = pd.DataFrame(records)
    mapping = {
        "table": table_name, "source_file": meta["source_file"],
        "assay_family": family, "source_version": version, "layout_detected": layout,
        "subject_column": str(subject_col) if subject_col is not None else None,
        "sample_column": str(sample_col) if sample_col is not None else None,
        "timepoint_column": str(time_col) if time_col is not None else None,
        "analyte_column": str(analyte_col) if analyte_col is not None else None,
        "value_column": str(value_col) if value_col is not None else None,
        "unit_column": str(unit_col) if unit_col is not None else None,
        "normalized_records": len(result),
        "rows_without_subject_id": int(result["subject_id"].isna().sum()) if not result.empty else 0,
        "rows_without_timepoint": int(result["timepoint"].isna().sum()) if not result.empty else 0,
        "rows_without_unit": int(result["unit"].isna().sum()) if not result.empty else 0,
    }
    return result, mapping

print("Normalization functions ready.")

Normalization functions ready.


In [7]:
# CELL 7 — normalize all submitted/transformed tables independently
normalized_frames = []
mapping_rows = []
normalization_errors = []
start = time.perf_counter()

for table_name, frame in tqdm(assay_tables.items(), desc="Normalizing source tables"):
    try:
        normalized, mapping = normalize_one_table(table_name, frame)
        mapping_rows.append(mapping)
        if not normalized.empty:
            normalized_frames.append(normalized)
        print(f"{table_name}: {len(normalized):,} biomarker record(s) [{mapping['layout_detected']}]")
    except Exception as exc:
        normalization_errors.append({"table": table_name, "error": f"{type(exc).__name__}: {exc}"})
        print(f"ERROR normalizing {table_name}: {exc}")

column_mapping_df = pd.DataFrame(mapping_rows)
normalization_error_df = pd.DataFrame(normalization_errors)
all_sources_df = pd.concat(normalized_frames, ignore_index=True) if normalized_frames else pd.DataFrame()

column_mapping_df.to_csv(TABLES_DIR / "osd575_column_mapping_and_normalization.csv", index=False)
normalization_error_df.to_csv(TABLES_DIR / "osd575_normalization_errors.csv", index=False)

if all_sources_df.empty:
    display(column_mapping_df)
    raise RuntimeError(
        "No numeric biomarker records were normalized. Inspect the column mapping table; "
        "the source tables may have an unhandled schema or measurement columns."
    )

all_sources_df.insert(0, "measurement_record_id", [f"OSD575-M{i:07d}" for i in range(1, len(all_sources_df) + 1)])
all_sources_path = PROCESSED_DIR / "osd575_measurements_long_all_sources.csv"
all_sources_df.to_csv(all_sources_path, index=False)

print(f"\nNormalized {len(all_sources_df):,} record(s) in {time.perf_counter() - start:.2f}s.")
print("All-source normalized table:", all_sources_path)
display(column_mapping_df)
display(all_sources_df.head(12))

Normalizing source tables:   0%|          | 0/7 [00:00<?, ?it/s]

LSDS-8_Comprehensive_Metabolic_Panel_CMP.upload_SUBMITTED: 509 biomarker record(s) [long]


Normalizing source tables:  29%|██▊       | 2/7 [00:00<00:00, 16.69it/s]

LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANSFORMED: 425 biomarker record(s) [wide]
LSDS-8_Multiplex_serum.cardiovascular.EvePanel_SUBMITTED: 252 biomarker record(s) [long]


Normalizing source tables:  71%|███████▏  | 5/7 [00:01<00:00,  2.42it/s]

LSDS-8_Multiplex_serum.immune.AlamarPanel_TRANSFORMED: 10,908 biomarker record(s) [wide]
LSDS-8_Multiplex_serum.immune.EvePanel_SUBMITTED: 1,988 biomarker record(s) [long]
LSDS-8_Multiplex_serum_cardiovascular_EvePanel_TRANSFORMED: 504 biomarker record(s) [wide]


Normalizing source tables: 100%|██████████| 7/7 [00:02<00:00,  2.66it/s]

LSDS-8_Multiplex_serum_immune_EvePanel_TRANSFORMED: 3,976 biomarker record(s) [wide]

Normalized 18,562 record(s) in 2.73s.
All-source normalized table: D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-nasa\data\processed\osd575_measurements_long_all_sources.csv


,table,source_file,assay_family,source_version,layout_detected,subject_column,sample_column,timepoint_column,analyte_column,value_column,unit_column,normalized_records,rows_without_subject_id,rows_without_timepoint,rows_without_unit
0,LSDS-8_Comprehensive_Metabolic_Panel_CMP.uploa...,LSDS-8_Comprehensive_Metabolic_Panel_CMP.uploa...,comprehensive_metabolic_panel,SUBMITTED,long,SUBJECT_ID,NaN,timepoint,ANALYTE,VALUE,UNITS,509,0,0,0
1,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,comprehensive_metabolic_panel,TRANSFORMED,wide,NaN,Sample Name,NaN,NaN,albumin_value_gram_per_deciliter,alkaline_phosphatase_value_units_per_liter,425,425,425,425
2,LSDS-8_Multiplex_serum.cardiovascular.EvePanel...,LSDS-8_Multiplex_serum.cardiovascular.EvePanel...,cardiovascular_multiplex,SUBMITTED,long,NaN,NaN,Timepoint,Analyte,Concentration,Unit,252,252,0,0
3,LSDS-8_Multiplex_serum.immune.AlamarPanel_TRAN...,LSDS-8_Multiplex_serum.immune.AlamarPanel_TRAN...,immune_alamar_multiplex,TRANSFORMED,wide,NaN,Sample ID,NaN,NaN,activin_a_concentration_npq,NaN,10908,10908,10908,10908
4,LSDS-8_Multiplex_serum.immune.EvePanel_SUBMITTED,LSDS-8_Multiplex_serum.immune.EvePanel_SUBMITT...,immune_eve_multiplex,SUBMITTED,long,NaN,NaN,Timepoint,Analyte,Concentration,Unit,1988,1988,0,0
5,LSDS-8_Multiplex_serum_cardiovascular_EvePanel...,LSDS-8_Multiplex_serum_cardiovascular_EvePanel...,cardiovascular_multiplex,TRANSFORMED,wide,NaN,Sample Name,NaN,NaN,a2_macroglobulin_concentration_nanogram_per_mi...,NaN,504,504,504,504
6,LSDS-8_Multiplex_serum_immune_EvePanel_TRANSFO...,LSDS-8_Multiplex_serum_immune_EvePanel_TRANSFO...,immune_eve_multiplex,TRANSFORMED,wide,NaN,Sample Name,NaN,NaN,6ckine_concentration_picogram_per_milliliter,NaN,3976,3976,3976,3976


,measurement_record_id,study_id,assay_family,assay_platform,analyte,value,raw_result_text,unit,subject_id,sample_id,timepoint,source_version,source_file,source_table,source_sheet,source_row_number,source_measurement_column,normalization_layout,numeric_parse_status
0,OSD575-M0000001,OSD-575,comprehensive_metabolic_panel,Quest Diagnostics CMP,GLUCOSE,83.00,83.0,mg/dL,C001,None,L-92,SUBMITTED,LSDS-8_Comprehensive_Metabolic_Panel_CMP.uploa...,LSDS-8_Comprehensive_Metabolic_Panel_CMP.uploa...,,2,VALUE,long,numeric
1,OSD575-M0000002,OSD-575,comprehensive_metabolic_panel,Quest Diagnostics CMP,UREA NITROGEN (BUN),25.00,25.0,mg/dL,C001,None,L-92,SUBMITTED,LSDS-8_Comprehensive_Metabolic_Panel_CMP.uploa...,LSDS-8_Comprehensive_Metabolic_Panel_CMP.uploa...,,3,VALUE,long,numeric
2,OSD575-M0000003,OSD-575,comprehensive_metabolic_panel,Quest Diagnostics CMP,CREATININE,1.24,1.24,mg/dL,C001,None,L-92,SUBMITTED,LSDS-8_Comprehensive_Metabolic_Panel_CMP.uploa...,LSDS-8_Comprehensive_Metabolic_Panel_CMP.uploa...,,4,VALUE,long,numeric
3,OSD575-M0000004,OSD-575,comprehensive_metabolic_panel,Quest Diagnostics CMP,eGFR NON-AFR. AMERICAN,73.00,73.0,mL/min/1.73m2,C001,None,L-92,SUBMITTED,LSDS-8_Comprehensive_Metabolic_Panel_CMP.uploa...,LSDS-8_Comprehensive_Metabolic_Panel_CMP.uploa...,,5,VALUE,long,numeric
4,OSD575-M0000005,OSD-575,comprehensive_metabolic_panel,Quest Diagnostics CMP,eGFR AFRICAN AMERICAN,85.00,85.0,mL/min/1.73m2,C001,None,L-92,SUBMITTED,LSDS-8_Comprehensive_Metabolic_Panel_CMP.uploa...,LSDS-8_Comprehensive_Metabolic_Panel_CMP.uploa...,,6,VALUE,long,numeric
5,OSD575-M0000006,OSD-575,comprehensive_metabolic_panel,Quest Diagnostics CMP,SODIUM,141.00,141.0,mmol/L,C001,None,L-92,SUBMITTED,LSDS-8_Comprehensive_Metabolic_Panel_CMP.uploa...,LSDS-8_Comprehensive_Metabolic_Panel_CMP.uploa...,,8,VALUE,long,numeric
6,OSD575-M0000007,OSD-575,comprehensive_metabolic_panel,Quest Diagnostics CMP,POTASSIUM,4.60,4.6,mmol/L,C001,None,L-92,SUBMITTED,LSDS-8_Comprehensive_Metabolic_Panel_CMP.uploa...,LSDS-8_Comprehensive_Metabolic_Panel_CMP.uploa...,,9,VALUE,long,numeric
7,OSD575-M0000008,OSD-575,comprehensive_metabolic_panel,Quest Diagnostics CMP,CHLORIDE,104.00,104.0,mmol/L,C001,None,L-92,SUBMITTED,LSDS-8_Comprehensive_Metabolic_Panel_CMP.uploa...,LSDS-8_Comprehensive_Metabolic_Panel_CMP.uploa...,,10,VALUE,long,numeric
8,OSD575-M0000009,OSD-575,comprehensive_metabolic_panel,Quest Diagnostics CMP,CARBON DIOXIDE,23.00,23.0,mmol/L,C001,None,L-92,SUBMITTED,LSDS-8_Comprehensive_Metabolic_Panel_CMP.uploa...,LSDS-8_Comprehensive_Metabolic_Panel_CMP.uploa...,,11,VALUE,long,numeric
9,OSD575-M0000010,OSD-575,comprehensive_metabolic_panel,Quest Diagnostics CMP,CALCIUM,9.70,9.7,mg/dL,C001,None,L-92,SUBMITTED,LSDS-8_Comprehensive_Metabolic_Panel_CMP.uploa...,LSDS-8_Comprehensive_Metabolic_Panel_CMP.uploa...,,12,VALUE,long,numeric


In [8]:
# CELL 8 — select one source version per assay family (without mixing versions)
if PRIMARY_SOURCE_POLICY not in {"prefer_transformed", "prefer_submitted"}:
    raise ValueError("PRIMARY_SOURCE_POLICY must be 'prefer_transformed' or 'prefer_submitted'.")

preferred_version = "TRANSFORMED" if PRIMARY_SOURCE_POLICY == "prefer_transformed" else "SUBMITTED"
secondary_version = "SUBMITTED" if preferred_version == "TRANSFORMED" else "TRANSFORMED"

selected_tables = []
selection_rows = []

for family, group in provenance_df.groupby("assay_family", dropna=False):
    if family == "unknown_assay_family":
        selection_rows.append({"assay_family": family, "selected_version": "NOT_SELECTED", "selected_tables": "", "reason": "Unknown assay family; manual review required."})
        continue
    versions = set(group["file_version"].astype(str))
    if preferred_version in versions:
        chosen = preferred_version
        reason = f"Selected configured preference: {preferred_version}."
    elif secondary_version in versions:
        chosen = secondary_version
        reason = f"Preferred version unavailable; fell back to {secondary_version}."
    elif "UNSPECIFIED" in versions:
        chosen = "UNSPECIFIED"
        reason = "Only unspecified source version available."
    else:
        chosen = sorted(versions)[0]
        reason = "Fallback to first available version; review provenance."

    chosen_tables = group.loc[group["file_version"] == chosen, "table"].tolist()
    selected_tables.extend(chosen_tables)
    selection_rows.append({
        "assay_family": family, "selected_version": chosen,
        "selected_tables": "; ".join(chosen_tables), "reason": reason,
    })

selection_df = pd.DataFrame(selection_rows)
selection_df.to_csv(TABLES_DIR / "osd575_primary_source_selection.csv", index=False)

table_keep = set(selected_tables)
primary_df = all_sources_df.loc[all_sources_df["source_table"].isin(table_keep)].copy()
primary_path = PROCESSED_DIR / "osd575_measurements_long_primary.csv"
primary_df.to_csv(primary_path, index=False)

display(selection_df)
print(f"\nPrimary integration table: {len(primary_df):,} records")
print("Saved:", primary_path)
print("Policy:", PRIMARY_SOURCE_POLICY)
print("Reminder: review the source-selection reasons before product use.")

,assay_family,selected_version,selected_tables,reason
0,cardiovascular_multiplex,TRANSFORMED,LSDS-8_Multiplex_serum_cardiovascular_EvePanel...,Selected configured preference: TRANSFORMED.
1,comprehensive_metabolic_panel,TRANSFORMED,LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANS...,Selected configured preference: TRANSFORMED.
2,immune_alamar_multiplex,TRANSFORMED,LSDS-8_Multiplex_serum.immune.AlamarPanel_TRAN...,Selected configured preference: TRANSFORMED.
3,immune_eve_multiplex,TRANSFORMED,LSDS-8_Multiplex_serum_immune_EvePanel_TRANSFO...,Selected configured preference: TRANSFORMED.



Primary integration table: 15,813 records
Saved: D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-nasa\data\processed\osd575_measurements_long_primary.csv
Policy: prefer_transformed
Reminder: review the source-selection reasons before product use.


In [9]:
# CELL 9 — validate canonical records and create sample/timepoint manifest

required_columns = [
    "measurement_record_id", "study_id", "assay_family", "analyte", "value", "unit",
    "subject_id", "sample_id", "timepoint", "source_version", "source_file",
    "source_table", "source_row_number", "source_measurement_column",
]
missing_required = [c for c in required_columns if c not in primary_df.columns]
if missing_required:
    raise AssertionError(f"Primary table is missing required columns: {missing_required}")

validation_rows = [
    {"check": "primary_records", "count": len(primary_df), "status": "INFO"},
    {"check": "records_missing_subject_id", "count": int(primary_df["subject_id"].isna().sum()), "status": "REVIEW" if primary_df["subject_id"].isna().any() else "PASS"},
    {"check": "records_missing_timepoint", "count": int(primary_df["timepoint"].isna().sum()), "status": "REVIEW" if primary_df["timepoint"].isna().any() else "PASS"},
    {"check": "records_missing_unit", "count": int(primary_df["unit"].isna().sum()), "status": "REVIEW" if primary_df["unit"].isna().any() else "PASS"},
    {"check": "non_numeric_or_censored_values", "count": int(primary_df["numeric_parse_status"].ne("numeric").sum()), "status": "REVIEW" if primary_df["numeric_parse_status"].ne("numeric").any() else "PASS"},
    {"check": "duplicate_family_subject_time_analyte_unit", "count": int(primary_df.duplicated(["assay_family", "subject_id", "timepoint", "analyte", "unit"], keep=False).sum()), "status": "REVIEW"},
    {"check": "negative_numeric_values", "count": int((pd.to_numeric(primary_df["value"], errors="coerce") < 0).sum()), "status": "REVIEW"},
]
validation_df = pd.DataFrame(validation_rows)
validation_df.to_csv(TABLES_DIR / "osd575_processing_validation.csv", index=False)
display(validation_df)

sample_manifest = (
    primary_df.groupby(["assay_family", "source_version", "subject_id", "sample_id", "timepoint"], dropna=False)
    .agg(
        measurement_count=("measurement_record_id", "count"),
        analyte_count=("analyte", "nunique"),
        unit_count=("unit", "nunique"),
        source_files=("source_file", lambda x: "; ".join(sorted(set(map(str, x))))),
    )
    .reset_index()
)
sample_manifest.to_csv(PROCESSED_DIR / "osd575_sample_timepoint_manifest.csv", index=False)
display(sample_manifest.head(30))
print("Sample/timepoint manifest saved.")

,check,count,status
0,primary_records,15813,INFO
1,records_missing_subject_id,15813,REVIEW
2,records_missing_timepoint,15813,REVIEW
3,records_missing_unit,15813,REVIEW
4,non_numeric_or_censored_values,0,PASS
5,duplicate_family_subject_time_analyte_unit,15813,REVIEW
6,negative_numeric_values,0,REVIEW


,assay_family,source_version,subject_id,sample_id,timepoint,measurement_count,analyte_count,unit_count,source_files
0,cardiovascular_multiplex,TRANSFORMED,NaN,C001_serum_L-3,NaN,18,18,0,LSDS-8_Multiplex_serum_cardiovascular_EvePanel...
1,cardiovascular_multiplex,TRANSFORMED,NaN,C001_serum_L-44,NaN,18,18,0,LSDS-8_Multiplex_serum_cardiovascular_EvePanel...
2,cardiovascular_multiplex,TRANSFORMED,NaN,C001_serum_L-92,NaN,18,18,0,LSDS-8_Multiplex_serum_cardiovascular_EvePanel...
3,cardiovascular_multiplex,TRANSFORMED,NaN,C001_serum_R+1,NaN,18,18,0,LSDS-8_Multiplex_serum_cardiovascular_EvePanel...
4,cardiovascular_multiplex,TRANSFORMED,NaN,C001_serum_R+194,NaN,18,18,0,LSDS-8_Multiplex_serum_cardiovascular_EvePanel...
5,cardiovascular_multiplex,TRANSFORMED,NaN,C001_serum_R+45,NaN,18,18,0,LSDS-8_Multiplex_serum_cardiovascular_EvePanel...
6,cardiovascular_multiplex,TRANSFORMED,NaN,C001_serum_R+82,NaN,18,18,0,LSDS-8_Multiplex_serum_cardiovascular_EvePanel...
7,cardiovascular_multiplex,TRANSFORMED,NaN,C002_serum_L-3,NaN,18,18,0,LSDS-8_Multiplex_serum_cardiovascular_EvePanel...
8,cardiovascular_multiplex,TRANSFORMED,NaN,C002_serum_L-44,NaN,18,18,0,LSDS-8_Multiplex_serum_cardiovascular_EvePanel...
9,cardiovascular_multiplex,TRANSFORMED,NaN,C002_serum_L-92,NaN,18,18,0,LSDS-8_Multiplex_serum_cardiovascular_EvePanel...


Sample/timepoint manifest saved.


In [10]:
# CELL 10 — analyte-level summaries (never aggregate across analytes or units)
# These are descriptive summaries only. Each assay family/analyte/unit/timepoint remains separate.
summary_source = primary_df.copy()
summary_source["value"] = pd.to_numeric(summary_source["value"], errors="coerce")
summary_source = summary_source.dropna(subset=["value"])

summary_group_cols = ["assay_family", "analyte", "unit", "timepoint"]
summary_df = (
    summary_source.groupby(summary_group_cols, dropna=False)["value"]
    .agg(observed_n="count", mean="mean", median="median", std="std", minimum="min", maximum="max")
    .reset_index()
)
summary_df.to_csv(TABLES_DIR / "osd575_analyte_timepoint_summary.csv", index=False)

# Participant-level values are retained as separate rows for longitudinal plotting/consumer use.
subject_summary = (
    summary_source.groupby(["assay_family", "subject_id", "timepoint", "analyte", "unit"], dropna=False)
    .agg(value_mean_within_subject_timepoint=("value", "mean"), replicate_count=("value", "count"))
    .reset_index()
)
subject_summary.to_csv(PROCESSED_DIR / "osd575_subject_timepoint_biomarkers.csv", index=False)

display(summary_df.head(40))
print("Saved analyte/timepoint summaries and subject-timepoint biomarker table.")

,assay_family,analyte,unit,timepoint,observed_n,mean,median,std,minimum,maximum
0,cardiovascular_multiplex,a2_macroglobulin_concentration_nanogram_per_mi...,NaN,NaN,28,1.694412e+06,1.651200e+06,7.794591e+05,5.314339e+05,3.051700e+06
1,cardiovascular_multiplex,a2_macroglobulin_percent,NaN,NaN,28,1.161439e+02,1.131819e+02,5.342823e+01,3.642727e+01,2.091796e+02
2,cardiovascular_multiplex,agp_concentration_nanogram_per_milliliter,NaN,NaN,28,2.377602e+06,2.467350e+06,9.566546e+05,9.427543e+05,4.564200e+06
3,cardiovascular_multiplex,agp_percent,NaN,NaN,28,1.106552e+02,1.148321e+02,4.452335e+01,4.387642e+01,2.124209e+02
4,cardiovascular_multiplex,crp_concentration_picogram_per_milliliter,NaN,NaN,28,1.972549e+07,1.232350e+07,2.058135e+07,1.289600e+06,8.361700e+07
5,cardiovascular_multiplex,crp_percent,NaN,NaN,28,1.113648e+02,6.957516e+01,1.161967e+02,7.280734e+00,4.720790e+02
6,cardiovascular_multiplex,fetuin_a36_concentration_nanogram_per_milliliter,NaN,NaN,28,2.515964e+05,2.712771e+05,6.765400e+04,1.257854e+05,3.739334e+05
7,cardiovascular_multiplex,fetuin_a36_percent,NaN,NaN,28,1.027137e+02,1.107483e+02,2.761959e+01,5.135163e+01,1.526575e+02
8,cardiovascular_multiplex,fibrinogen_concentration_nanogram_per_milliliter,NaN,NaN,28,3.859966e+03,3.875635e+03,1.032881e+03,1.999880e+03,6.533180e+03
9,cardiovascular_multiplex,fibrinogen_percent,NaN,NaN,28,9.143446e+01,9.180563e+01,2.446677e+01,4.737294e+01,1.547573e+02


Saved analyte/timepoint summaries and subject-timepoint biomarker table.


In [11]:
# CELL 11 — generate processing report

def markdown_table(frame, max_rows=35):
    if frame is None or frame.empty:
        return "_No records available._"
    view = frame.head(max_rows).fillna("")
    lines = ["| " + " | ".join(map(str, view.columns)) + " |", "| " + " | ".join(["---"] * len(view.columns)) + " |"]
    for _, row in view.iterrows():
        vals = [str(v).replace("|", "/").replace("\n", " ") for v in row.tolist()]
        lines.append("| " + " | ".join(vals) + " |")
    return "\n".join(lines)

family_counts = primary_df.groupby("assay_family").agg(records=("measurement_record_id", "count"), analytes=("analyte", "nunique"), subjects=("subject_id", "nunique"), timepoints=("timepoint", "nunique")).reset_index()
version_counts = all_sources_df.groupby(["assay_family", "source_version"]).size().rename("records").reset_index()

report = f"# OSD-575 Biomarker Processing Report\n\n"
report += f"- Workstream: `{WORKSTREAM_ROOT}`\n- Primary source policy: `{PRIMARY_SOURCE_POLICY}`\n- Source tables loaded: {len(assay_tables)}\n- All-source normalized records: {len(all_sources_df):,}\n- Primary normalized records: {len(primary_df):,}\n- Distinct participants identified in primary table: {primary_df['subject_id'].nunique(dropna=True)}\n- Runtime: {time.perf_counter() - start:.2f} seconds (normalization timing; notebook total may be longer)\n\n"
report += "## Primary source selection\n\n" + markdown_table(selection_df) + "\n\n"
report += "## Primary table coverage\n\n" + markdown_table(family_counts) + "\n\n"
report += "## Source-version coverage (all normalized sources)\n\n" + markdown_table(version_counts) + "\n\n"
report += "## Validation checks\n\n" + markdown_table(validation_df) + "\n\n"
report += "## Column mapping\n\n" + markdown_table(column_mapping_df) + "\n\n"
report += "## Interpretation and use safeguards\n\n"
report += "- Submitted and transformed exports are normalized separately; do not count both as independent biological observations.\n"
report += "- The primary table follows the configured source policy. Review `osd575_primary_source_selection.csv` and the original EDA comparison before using it in a product.\n"
report += "- Units are never inferred from another assay or biomarker. Missing units remain missing.\n"
report += "- Rows without subject or timepoint mapping are retained but must be resolved before longitudinal person-level analysis.\n"
report += "- Repeated measurements, duplicates and negative values are flagged for inspection, not automatically deleted.\n"
report += "- This dataset represents four Inspiration4 crew members and a limited set of serum collection timepoints; it is not sufficient to validate population-wide astronaut-health predictions.\n"
report += "- This notebook performs data normalization and descriptive aggregation only; it does not produce diagnoses or clinical alert thresholds.\n\n"
report += "## Output files\n\n"
for p in [all_sources_path, primary_path, PROCESSED_DIR / "osd575_sample_timepoint_manifest.csv", PROCESSED_DIR / "osd575_subject_timepoint_biomarkers.csv", TABLES_DIR / "osd575_analyte_timepoint_summary.csv", TABLES_DIR / "osd575_processing_validation.csv", TABLES_DIR / "osd575_column_mapping_and_normalization.csv", TABLES_DIR / "osd575_primary_source_selection.csv"]:
    report += f"- `{p.relative_to(WORKSTREAM_ROOT)}`\n"

report_path = REPORTS_DIR / "osd575_biomarker_processing_report.md"
report_path.write_text(report, encoding="utf-8")
print("Report saved:", report_path)
print("\n" + report[:4500])

Report saved: D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-nasa\reports\osd575_biomarker_processing_report.md

# OSD-575 Biomarker Processing Report

- Workstream: `D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-nasa`
- Primary source policy: `prefer_transformed`
- Source tables loaded: 7
- All-source normalized records: 18,562
- Primary normalized records: 15,813
- Distinct participants identified in primary table: 0
- Runtime: 2.94 seconds (normalization timing; notebook total may be longer)

## Primary source selection

| assay_family | selected_version | selected_tables | reason |
| --- | --- | --- | --- |
| cardiovascular_multiplex | TRANSFORMED | LSDS-8_Multiplex_serum_cardiovascular_EvePanel_TRANSFORMED | Selected configured preference: TRANSFORMED. |
| comprehensive_metabolic_panel | TRANSFORMED | LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANSFORMED | Selected configured preference: TRANSFORMED. |
| immun

In [12]:
# CELL 12 — final output check
output_paths = [
    PROCESSED_DIR / "osd575_measurements_long_all_sources.csv",
    PROCESSED_DIR / "osd575_measurements_long_primary.csv",
    PROCESSED_DIR / "osd575_sample_timepoint_manifest.csv",
    PROCESSED_DIR / "osd575_subject_timepoint_biomarkers.csv",
    TABLES_DIR / "osd575_analyte_timepoint_summary.csv",
    TABLES_DIR / "osd575_processing_validation.csv",
    TABLES_DIR / "osd575_column_mapping_and_normalization.csv",
    TABLES_DIR / "osd575_primary_source_selection.csv",
    REPORTS_DIR / "osd575_biomarker_processing_report.md",
]

output_check = pd.DataFrame([
    {"file": str(path.relative_to(WORKSTREAM_ROOT)), "exists": path.exists(), "size_bytes": path.stat().st_size if path.exists() else 0}
    for path in output_paths
])
display(output_check)

if not output_check["exists"].all():
    raise RuntimeError("One or more expected outputs are missing. Inspect earlier cell errors.")

print("\nOSD-575 biomarker processing completed successfully.")
print("Next practical step: connect this canonical table to the application's biomarker observation model.")

,file,exists,size_bytes
0,data\processed\osd575_measurements_long_all_so...,True,5532480
1,data\processed\osd575_measurements_long_primar...,True,4838491
2,data\processed\osd575_sample_timepoint_manifes...,True,13781
3,data\processed\osd575_subject_timepoint_biomar...,True,44832
4,reports\tables\osd575_analyte_timepoint_summar...,True,74443
5,reports\tables\osd575_processing_validation.csv,True,294
6,reports\tables\osd575_column_mapping_and_norma...,True,1816
7,reports\tables\osd575_primary_source_selection...,True,603
8,reports\osd575_biomarker_processing_report.md,True,6006



OSD-575 biomarker processing completed successfully.
Next practical step: connect this canonical table to the application's biomarker observation model.
